# Automatización web con Computer Use

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/computer-use-operator/02-automatizacion-web.ipynb)

> ⚠️ Requiere entorno con display virtual. Celdas ejecutables como demo de arquitectura.

In [ ]:
!pip install anthropic -q

In [ ]:
import anthropic
client = anthropic.Anthropic()

## ¿Cuándo usar Computer Use vs Playwright?

| Situación | Recomendado |
|-----------|-------------|
| DOM limpio y estático | Playwright / BeautifulSoup |
| SPA que cambia frecuentemente | Computer Use |
| Portal ERP legacy sin API | Computer Use |
| Formulario con CAPTCHA visual | Computer Use |
| Web scraping simple | requests + BS4 |

## Patrón de scraping visual

In [ ]:
def construir_instruccion_scraping(url: str, campos: list[str]) -> str:
    """Genera una instrucción de scraping visual para Computer Use."""
    campos_str = '\n'.join(f'  - {campo}' for campo in campos)
    return f"""Ve a {url} y extrae los siguientes campos:
{campos_str}

Devuelve los datos en formato JSON: {{"campo": "valor", ...}}
Si algún campo no está visible, usa null como valor."""

# Ejemplos de instrucciones
instrucciones = [
    construir_instruccion_scraping(
        'https://erp.ejemplo.com/facturas/12345',
        ['numero_factura', 'fecha', 'proveedor', 'importe_total', 'estado']
    ),
    construir_instruccion_scraping(
        'https://portal.proveedor.com/pedido/67890',
        ['estado_envio', 'fecha_entrega_estimada', 'numero_seguimiento']
    ),
]

for i, inst in enumerate(instrucciones, 1):
    print(f'Instrucción {i}:')
    print(inst)
    print()

## Verificación visual rápida con Haiku

In [ ]:
import base64
from pathlib import Path

def verificar_elemento_presente(imagen_b64: str, descripcion: str) -> bool:
    """
    Usa Haiku (barato y rápido) para verificar si un elemento
    visual está presente en la pantalla.
    """
    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=16,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/png', 'data': imagen_b64}},
                {'type': 'text', 'text': f'¿Está visible "{descripcion}" en la imagen? Responde solo SÍ o NO.'},
            ],
        }],
    )
    return 'SÍ' in response.content[0].text.upper()

print('Función lista. Usa una imagen real en producción.')
print('\nPatrón de uso:')
print('''imagen = base64.standard_b64encode(Path("/tmp/screenshot.png").read_bytes()).decode()
if verificar_elemento_presente(imagen, "botón Guardar"):
    print("Botón encontrado, proceder con el clic")
else:
    print("Botón no visible, esperar o reintentar")''')

## Manejo de errores y reintentos

In [ ]:
import time
from typing import Callable, TypeVar

T = TypeVar('T')

def con_reintentos(fn: Callable[[], T], max_intentos: int = 3, espera_base: float = 2.0) -> T:
    """Ejecuta una función con reintentos exponenciales."""
    ultimo_error = None
    for intento in range(max_intentos):
        try:
            return fn()
        except Exception as e:
            ultimo_error = e
            espera = espera_base ** intento
            print(f'Intento {intento+1} fallido: {e}. Esperando {espera:.0f}s...')
            time.sleep(espera)
    raise RuntimeError(f'Fallido tras {max_intentos} intentos: {ultimo_error}')

# Demo: simular reintentos
intentos = [0]

def operacion_inestable():
    intentos[0] += 1
    if intentos[0] < 3:
        raise ConnectionError(f'Timeout en intento {intentos[0]}')
    return f'Éxito en intento {intentos[0]}'

resultado = con_reintentos(operacion_inestable, max_intentos=3, espera_base=0.5)
print(f'Resultado: {resultado}')